In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib


In [2]:
df = pd.read_csv("synthetic_groundnut_digital_twin_dataset.csv")

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42
)


In [3]:
residual_model = joblib.load("dt_residual_model.pkl")


In [4]:
def digital_twin_baseline(row):
    return (
        700
        + 200 * row["seasonal_stress"]
        + 150 * row["harvest_index"]
    )


In [5]:
y_true = test_df["yield_kg_per_ha"]

dt_preds = test_df.apply(digital_twin_baseline, axis=1)

X_test = test_df.drop(
    columns=["yield_kg_per_ha"]
)

residual_preds = residual_model.predict(X_test)

y_pred = dt_preds + residual_preds


In [6]:
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

mae, r2


(10.651113245990308, 0.988272382282696)

In [7]:
real_data = pd.DataFrame([
    {
        "seasonal_stress": 0.72,
        "LAI_max": 5.1,
        "leaf_growth_rate": 0.07,
        "RUE": 1.4,
        "harvest_index": 0.36,
        "yield_kg_per_ha": 740
    }
])


In [8]:
dt_pred = digital_twin_baseline(real_data.iloc[0])
residual = residual_model.predict(
    real_data.drop(columns=["yield_kg_per_ha"])
)[0]

final_pred = dt_pred + residual
final_pred


np.float64(556.6995239503697)

In [9]:
error = real_data["yield_kg_per_ha"].iloc[0] - final_pred
error


np.float64(183.30047604963033)